# SAM3 Bedroom Dataset Pipeline
This notebook prepares the bedroom photos in Google Drive for instance segmentation annotations using the [`Sam3Model`](https://huggingface.co/docs/transformers/main/en/model_doc/sam3) and exports COCO-formatted metadata to the processed folder. Follow the cells sequentially when running on Google Colab with GPU acceleration.

## Execution Plan
1. Verify runtime (GPU, Python, CUDA) and install all dependencies missing from vanilla Colab.
2. Authenticate with Hugging Face (token stored in env), mount Google Drive, and define raw/processed bedroom paths.
3. Enumerate raw `bedroom_images`, add helper utilities (loading, mask conversion, prompt grid generation).
4. Download `facebook/sam3` processor/model with the HF token, configure device placement, and warm up inference.
5. For every image, run SAM3 once with eight prompts (one per target class) and collate COCO annotations.
6. Persist masks + JSON under `processed-images/bedroom`, review stats, and follow up with manual QA.

In [ ]:
# Root of the shared project folder on Google Drive.
# Team members mounted it at different places in their Drives, so all paths
# below are built from this single constant. Change it to match your setup.
DRIVE_ROOT = "/content/drive/MyDrive/furniture-retrieval"

In [ ]:
import os
import platform
import torch
from datetime import datetime

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
runtime_report = {
    "timestamp": datetime.utcnow().isoformat(timespec="seconds") + "Z",
    "python": platform.python_version(),
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
}
runtime_report

In [ ]:
%%capture
!pip install --upgrade pip
!pip install --quiet --upgrade "transformers>=4.45.0.dev0" accelerate datasets pillow opencv-python pycocotools tqdm

In [ ]:
if "HF_TOKEN" not in os.environ or not os.environ["HF_TOKEN"].strip():
    hf_token = getpass("Enter your Hugging Face token (with access to facebook/sam3): ")
    os.environ["HF_TOKEN"] = hf_token.strip()
else:
    hf_token = os.environ["HF_TOKEN"]

login(token=hf_token, add_to_git_credential=False)
print("✓ Hugging Face token configured in this runtime")

In [ ]:
import json
from getpass import getpass
from pathlib import Path
from typing import Dict, List, Tuple


import cv2
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from huggingface_hub import login

from transformers import Sam3Processor, Sam3Model

In [ ]:
try:
    from google.colab import drive  # type: ignore
except ImportError:
    drive = None
    print("google.colab not detected; please run this cell on Colab or mount manually.")
else:
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")
    else:
        print("Google Drive seems to be mounted already.")

In [ ]:
DRIVE_ROOT = Path(DRIVE_ROOT)
RAW_IMAGES_DIR = DRIVE_ROOT / "raw-images" / "bedroom_images"
PROCESSED_DIR = DRIVE_ROOT / "processed-images" / "bedroom"
MASK_DIR = PROCESSED_DIR / "masks"
COCO_PATH = PROCESSED_DIR / "annotations.json"

for folder in [RAW_IMAGES_DIR, PROCESSED_DIR, MASK_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

len(list(RAW_IMAGES_DIR.glob("*.jpg")))

In [ ]:
SUPPORTED_EXTS = (".jpg", ".jpeg", ".png", ".webp")


def list_images(root: Path) -> List[Path]:
    return sorted(p for p in root.iterdir() if p.suffix.lower() in SUPPORTED_EXTS)

def load_image(path: Path) -> Image.Image:
    return Image.open(path).convert("RGB")

raw_images = list_images(RAW_IMAGES_DIR)
print(f"Found {len(raw_images)} raw bedroom images")

In [ ]:
CATEGORIES = [
    {"id": 1, "name": "bed", "supercategory": "bedroom"},
    {"id": 2, "name": "cabinet", "supercategory": "bedroom"},
    {"id": 3, "name": "carpet", "supercategory": "bedroom"},
    {"id": 4, "name": "chair", "supercategory": "bedroom"},
    {"id": 5, "name": "lamp", "supercategory": "bedroom"},
    {"id": 6, "name": "shelf", "supercategory": "bedroom"},
    {"id": 7, "name": "sofa", "supercategory": "bedroom"},
    {"id": 8, "name": "table", "supercategory": "bedroom"},
]

In [ ]:
model_name = "facebook/sam3"

hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN is not set. Please run the authentication cell and rerun this cell.")

processor = Sam3Processor.from_pretrained(model_name, token=hf_token)
model = Sam3Model.from_pretrained(model_name, token=hf_token).to(device)
model.eval()

# Pre-compute text embeddings for all 8 categories
text_embeddings = {}
for category in CATEGORIES:
    text_inputs = processor(text=category["name"], return_tensors="pt").to(device)
    with torch.no_grad():
        text_embeddings[category["id"]] = {
            "embeds": model.get_text_features(**text_inputs),
            "attention_mask": text_inputs.attention_mask,
        }

print(f"Loaded {model_name} on {device} with {len(text_embeddings)} text embeddings precomputed")

In [ ]:
def run_sam3_with_text(image: Image.Image, text_embeds, attention_mask):
    """Run SAM3 inference with pre-computed text embeddings."""
    img_inputs = processor(images=image, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model(
            pixel_values=img_inputs.pixel_values,
            text_embeds=text_embeds,
            attention_mask=attention_mask,
        )
    
    # Post-process to get instance segmentation results
    results = processor.post_process_instance_segmentation(
        outputs,
        threshold=0.5,
        mask_threshold=0.5,
        target_sizes=img_inputs.get("original_sizes").tolist()
    )[0]
    
    return results

mask_threshold = 0.5
score_threshold = 0.5
min_area_px = 10

In [ ]:
# Initialize COCO format with metadata
COCO = {
    "info": {
        "description": "SAM3 Generated Bedroom Dataset",
        "version": "1.0",
        "year": datetime.now().year,
        "date_created": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    },
    "licenses": [],
    "images": [],
    "annotations": [],
    "categories": CATEGORIES
}

annotation_id = 1

# Process each image with all 8 categories
for image_id, image_path in enumerate(tqdm(raw_images, desc="Processing images"), start=1):
    image = load_image(image_path)
    width, height = image.size

    # Add image metadata
    COCO["images"].append({
        "id": image_id,
        "file_name": image_path.name,
        "width": width,
        "height": height,
    })

    # Process each category separately
    for category in CATEGORIES:
        category_id = category["id"]
        category_name = category["name"]
        
        # Get pre-computed text embeddings for this category
        text_embeds = text_embeddings[category_id]["embeds"]
        attention_mask = text_embeddings[category_id]["attention_mask"]
        
        # Run SAM3 inference
        results = run_sam3_with_text(image, text_embeds, attention_mask)
        
        # Process each detected mask
        for mask, score in zip(results['masks'], results['scores']):
            if score < score_threshold:
                continue
            
            # Convert mask to numpy array
            mask_np = mask.cpu().numpy().astype(np.uint8)
            
            # Find contours
            contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                # Calculate area
                area = cv2.contourArea(contour)
                
                # Skip small contours
                if area < min_area_px:
                    continue
                
                # Convert contour to segmentation format
                segmentation = contour.flatten().tolist()
                
                # Calculate bounding box
                x, y, w, h = cv2.boundingRect(contour)
                
                # Save mask image
                mask_path = MASK_DIR / f"{image_path.stem}_{category_name}_ann_{annotation_id}.png"
                cv2.imwrite(str(mask_path), mask_np * 255)
                
                # Add annotation to COCO
                COCO["annotations"].append({
                    "id": annotation_id,
                    "image_id": image_id,
                    "category_id": category_id,
                    "segmentation": [segmentation],
                    "area": float(area),
                    "bbox": [int(x), int(y), int(w), int(h)],
                    "iscrowd": 0,
                    "score": float(score),
                    "mask_path": str(mask_path.relative_to(DRIVE_ROOT)),
                })
                annotation_id += 1

print(f"\nGenerated {len(COCO['annotations'])} annotations across {len(COCO['images'])} images with {len(CATEGORIES)} classes")

In [ ]:
with open(COCO_PATH, "w", encoding="utf-8") as f:
    json.dump(COCO, f, indent=2)

print(f"\nCOCO annotations saved to: {COCO_PATH}")
print(f"Total images: {len(COCO['images'])}")
print(f"Total annotations: {len(COCO['annotations'])}")
print(f"Total categories: {len(COCO['categories'])}")

### Next Steps
- Review the per-class masks under `processed-images/bedroom/masks` to validate SAM3 detected the correct objects for each text prompt.
- Adjust `score_threshold` or `min_area_px` in the inference cell if you see too many false positives or need to filter small detections.
- Use the generated `annotations.json` (in COCO format) directly with your training pipeline or import it into a labeling tool for manual review.